ridge/lasso classical ML-also going to serve as a baseline

importation of libraries

In [5]:
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LogisticRegressionCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    f1_score, average_precision_score,
    confusion_matrix, classification_report
)

reproducibility seed

In [6]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

forecasting horizons in hours

In [7]:
HORIZONS = {
    "15min": 1,
    "30min": 2,
    "60min": 4,
}

loading, cleaning and sorting the data.

In [8]:
DATA_PATH = "traffic_features org.csv"
df = pd.read_csv(DATA_PATH)
print(f"  Raw shape: {df.shape}")
 

# Replacing ??? with NaN to let pandas handle it uniformly in all later steps.
df.replace("???", np.nan, inplace=True)
 

# setting timestamp to a proper format to support later computations and sorting of rows
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
 
# Dropping rows where the timestamp itself is missing.
n_before = len(df)
df.dropna(subset=["timestamp"], inplace=True)
print(f"  Dropped {n_before - len(df)} rows with missing timestamp")
 
# Sorting chronologically.
df.sort_values("timestamp", inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"  Sorted {len(df)} rows chronologically")
 
#  Parse numeric columns like avg_speed,vechile count, hour and weekday.
df["avg_speed"] = pd.to_numeric(df["avg_speed"], errors="coerce")
 
df["vehicle_count"] = pd.to_numeric(df["vehicle_count"], errors="coerce")
 
df["hour"]    = pd.to_numeric(df["hour"],    errors="coerce")
df["weekday"] = pd.to_numeric(df["weekday"], errors="coerce")
 
#  Cleaning target_congestion_level by converting values first to float, the to nullable Int8 inorder to perserve the nan rows
df["target_congestion_level"] = pd.to_numeric(
    df["target_congestion_level"], errors="coerce"
)
# Valid classes are 0-4 only. Anything outside that range is corrupt.
invalid_mask = ~df["target_congestion_level"].isin([0, 1, 2, 3, 4]) & \
               df["target_congestion_level"].notna()
df.loc[invalid_mask, "target_congestion_level"] = np.nan
print(f"  target_congestion_level: "
      f"{df['target_congestion_level'].notna().sum()} valid labels, "
      f"{df['target_congestion_level'].isna().sum()} missing/corrupt")
 
print(f"  Final clean shape: {df.shape}")

  Raw shape: (1464, 16)
  Dropped 5 rows with missing timestamp
  Sorted 1459 rows chronologically
  target_congestion_level: 1453 valid labels, 6 missing/corrupt
  Final clean shape: (1459, 16)


building lag features to give the ridge model memory of past values of the target explicitly as imput features

In [9]:

for lag in [2, 3, 6, 24]:
    col_name = f"lag{lag}"
    df[col_name] = df["vehicle_count"].shift(lag)
    n_valid = df[col_name].notna().sum()
    print(f"  Created {col_name}: {n_valid} non-null values")
 
# since lag1 and roll3 already exists in the CSV, we keep it asit is.
print("  lag1 and roll3 retained from original CSV")

  Created lag2: 1447 non-null values
  Created lag3: 1446 non-null values
  Created lag6: 1443 non-null values
  Created lag24: 1425 non-null values
  lag1 and roll3 retained from original CSV


Encodeing categorical features of text columns into numbers to support the model. emphasis on using sin and cos to encode the hour and weekday feature is so that hours 23 and 0 are understood as adjacent points by the model,otherwise, would be treating them as distant ie misunderstanding the data

In [10]:
# Cyclical encoding for hour and weekday 
df["hour_sin"]     = np.sin(2 * np.pi * df["hour"]    / 24)
df["hour_cos"]     = np.cos(2 * np.pi * df["hour"]    / 24)
df["weekday_sin"]  = np.sin(2 * np.pi * df["weekday"] / 7)
df["weekday_cos"]  = np.cos(2 * np.pi * df["weekday"] / 7)
print("  Created hour_sin, hour_cos, weekday_sin, weekday_cos")
 
# One-hot encoding weather 
# NaN values become a separate "weather_unknown" column so the model knows data was absent.
df["weather_filled"] = df["weather"].fillna("Unknown")
weather_dummies = pd.get_dummies(
    df["weather_filled"], prefix="weather", drop_first=False, dtype=float
)
df = pd.concat([df, weather_dummies], axis=1)
print(f"  Weather dummies: {list(weather_dummies.columns)}")
 
#  One-hot encoding road_condition
df["road_condition_filled"] = df["road_condition"].fillna("Unknown")
road_dummies = pd.get_dummies(
    df["road_condition_filled"], prefix="road", drop_first=False, dtype=float
)
df = pd.concat([df, road_dummies], axis=1)
print(f"  Road condition dummies: {list(road_dummies.columns)}")
 
#  Binary event flag ie any non-null, non-empty value in the event column means an event is happening.
df["event_flag"] = df["event"].notna().astype(float)
print(f"  event_flag: {int(df['event_flag'].sum())} rows have an active event")
 
#Binary sensor reliability flag ie 1 = sensor reporting normally and is ok, 0 = degraded/failed/unknown
df["sensor_ok"] = (df["sensor_status"] == "OK").astype(float)
print(f"  sensor_ok: {int(df['sensor_ok'].sum())} rows with healthy sensor")
 
# is_holiday and school_in_session are already binary ints,performing this action just to be sure.
df["is_holiday"]  = pd.to_numeric(df["is_holiday"],  errors="coerce")
df["school_in_session"]= pd.to_numeric(df["school_in_session"],errors="coerce")
print("  is_holiday and school_in_session confirmed numeric")

  Created hour_sin, hour_cos, weekday_sin, weekday_cos
  Weather dummies: ['weather_Clear', 'weather_Foggy', 'weather_Rainy', 'weather_Unknown']
  Road condition dummies: ['road_Good', 'road_Moderate', 'road_Poor', 'road_Unknown']
  event_flag: 1085 rows have an active event
  sensor_ok: 501 rows with healthy sensor
  is_holiday and school_in_session confirmed numeric


implementing rolling-origin temporal splitting without shuffling or random sampling. why this? beacuse our dataset has high autocorrelation and shuffling would cause temporal leakage(futier values apearing in the training dataset causing the model to "cheat")

In [11]:

n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)
 
train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()
 
print(f"  Total rows  : {n}")
print(f"  Train rows  : {len(train_df)}  "
      f"({train_df['timestamp'].min().date()} → {train_df['timestamp'].max().date()})")
print(f"  Val rows    : {len(val_df)}  "
      f"({val_df['timestamp'].min().date()} → {val_df['timestamp'].max().date()})")
print(f"  Test rows   : {len(test_df)}  "
      f"({test_df['timestamp'].min().date()} → {test_df['timestamp'].max().date()})")

  Total rows  : 1459
  Train rows  : 1021  (2023-01-01 → 2023-01-12)
  Val rows    : 219  (2023-01-12 → 2023-01-14)
  Test rows   : 219  (2023-01-14 → 2023-01-16)


imputing missing values using training statistics beacause val/test statistics would leak future information.
for lag and roll features, i group median by hour_bin(this groups hours into 6 four-hour blocks so each group has enough training rows to compute a stable median) and weekday
then for avg_speed, I use global training meadian and i fill with zeros for is_holiday and school_in_session.

In [12]:
def group_impute(target_df, train_ref, col, group_cols):
    
    group_medians = (
        train_ref.groupby(group_cols)[col]
        .median()
        .reset_index()
        .rename(columns={col: f"{col}_med"})
    )
    merged = target_df[group_cols + [col]].merge(
        group_medians, on=group_cols, how="left"
    )
    global_med = train_ref[col].median()
    filled = target_df[col].copy()
    missing_mask = filled.isna()
    group_fill   = merged[f"{col}_med"].values
    global_fill  = global_med
 
    filled[missing_mask] = np.where(
        ~np.isnan(group_fill[missing_mask]),
        group_fill[missing_mask],
        global_fill
    )
    return filled
 
 
# Creating the 4-hour block grouping variable (0–5) for better group coverage
for split in [train_df, val_df, test_df]:
    split["hour_bin"] = (split["hour"] // 4).fillna(0).astype(int)
 
group_keys = ["hour_bin", "weekday"]
 
lag_cols_to_impute = ["lag1", "lag2", "lag3", "lag6", "lag24", "roll3"]
 
for col in lag_cols_to_impute:
    before_train = train_df[col].isna().sum()
    before_val   = val_df[col].isna().sum()
    before_test  = test_df[col].isna().sum()
 
    train_df[col] = group_impute(train_df, train_df, col, group_keys)
    val_df[col]   = group_impute(val_df,   train_df, col, group_keys)
    test_df[col]  = group_impute(test_df,  train_df, col, group_keys)
 
    print(f"  {col}: filled {before_train} train / {before_val} val / "
          f"{before_test} test missing values")
 
# avg_speed: filling with global training median
speed_med = train_df["avg_speed"].median()
for split in [train_df, val_df, test_df]:
    split["avg_speed"] = split["avg_speed"].fillna(speed_med)
print(f"  avg_speed: filled with training median ({speed_med:.2f})")
 
#  fill with 0 for is_holiday and school_in_session
for col in ["is_holiday", "school_in_session"]:
    for split in [train_df, val_df, test_df]:
        split[col] = split[col].fillna(0)
print("  is_holiday and school_in_session: filled with 0")

  lag1: filled 12 train / 2 val / 5 test missing values
  lag2: filled 8 train / 2 val / 2 test missing values
  lag3: filled 9 train / 2 val / 2 test missing values
  lag6: filled 12 train / 2 val / 2 test missing values
  lag24: filled 30 train / 2 val / 2 test missing values
  roll3: filled 29 train / 6 val / 9 test missing values
  avg_speed: filled with training median (40.01)
  is_holiday and school_in_session: filled with 0


defining the feature matrix ie he final list of colums going to be fed to the models.

In [13]:
FEATURE_COLS = [
    # Auto-regressive lags to aid in traffic memory
    "lag1", "lag2", "lag3", "lag6", "lag24", "roll3",
    "avg_speed",
 
    # Cyclical temporal encodings
    "hour_sin", "hour_cos",
    "weekday_sin", "weekday_cos",
 
    # Binary calendar flags.
    "is_holiday", "school_in_session",
 
    # Exogenous: weather (one-hot) 
    "weather_Clear", "weather_Foggy", "weather_Rainy", "weather_Unknown",
 
    # Exogenous: road condition (one-hot) 
    "road_Good", "road_Moderate", "road_Poor", "road_Unknown",
 
    # Exogenous: event and sensor
    "event_flag", "sensor_ok",
]
 
# Keep only columns that actually exist after encoding(get_dummies column names depend on the values present in the data)
FEATURE_COLS = [c for c in FEATURE_COLS if c in train_df.columns]
print(f"  Feature matrix: {len(FEATURE_COLS)} columns")
print(f"  Columns: {FEATURE_COLS}")
 
# Regression target
REG_TARGET = "vehicle_count"
 
# Classification target
CLF_TARGET = "target_congestion_level"

  Feature matrix: 23 columns
  Columns: ['lag1', 'lag2', 'lag3', 'lag6', 'lag24', 'roll3', 'avg_speed', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'is_holiday', 'school_in_session', 'weather_Clear', 'weather_Foggy', 'weather_Rainy', 'weather_Unknown', 'road_Good', 'road_Moderate', 'road_Poor', 'road_Unknown', 'event_flag', 'sensor_ok']


fitting the scaler on training data only(for now-i will apply to all three splits later to prevent information leaking).implementing this standardscaler puts every feature on equal footing.

In [14]:

scaler = StandardScaler()
 
X_train = scaler.fit_transform(train_df[FEATURE_COLS])
X_val   = scaler.transform(val_df[FEATURE_COLS])
X_test  = scaler.transform(test_df[FEATURE_COLS])
 
print(f"  Scaler fitted on {X_train.shape[0]} training rows, "
      f"{X_train.shape[1]} features")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_val   shape: {X_val.shape}")
print(f"  X_test  shape: {X_test.shape}")
 
# ── Regression targets — mask rows where vehicle_count is NaN ─────────────
# 10 rows have missing vehicle_count; we must exclude them from training.
# keeping a full-length array (with NaN) for evaluation  so the horizon-shift. this logic  lines up correctly in Section 10, then NaN will be dropped after shifting.
reg_train_mask = train_df[REG_TARGET].notna()
reg_val_mask   = val_df[REG_TARGET].notna()
 
X_train_reg = X_train[reg_train_mask.values]
y_train_reg = train_df.loc[reg_train_mask, REG_TARGET].values
 
X_val_reg   = X_val[reg_val_mask.values]
y_val_reg   = val_df.loc[reg_val_mask, REG_TARGET].values
 
# Full-length test array (preserves NaN) used in horizon-shift evaluation
y_test_reg_full = test_df[REG_TARGET].values
 
# ── Classification targets — only rows where label is 0-4 ──────────────────
# keeping a boolean mask so we can subset X arrays correctly.
clf_train_mask = train_df[CLF_TARGET].notna()
clf_val_mask   = val_df[CLF_TARGET].notna()
clf_test_mask  = test_df[CLF_TARGET].notna()
 
X_train_clf = X_train[clf_train_mask.values]
y_train_clf = train_df.loc[clf_train_mask, CLF_TARGET].astype(int).values
 
X_val_clf   = X_val[clf_val_mask.values]
y_val_clf   = val_df.loc[clf_val_mask, CLF_TARGET].astype(int).values
 
X_test_clf  = X_test[clf_test_mask.values]
y_test_clf  = test_df.loc[clf_test_mask, CLF_TARGET].astype(int).values
 
print(f"\n  Regression  — train: {len(y_train_reg)}, "
      f"val: {len(y_val_reg)}, test: {len(y_test_reg_full)} (full incl. NaN)")
print(f"  Classification — train: {len(y_train_clf)}, "
      f"val: {len(y_val_clf)}, test: {len(y_test_clf)}")

  Scaler fitted on 1021 training rows, 23 features
  X_train shape: (1021, 23)
  X_val   shape: (219, 23)
  X_test  shape: (219, 23)

  Regression  — train: 1015, val: 217, test: 219 (full incl. NaN)
  Classification — train: 1016, val: 219, test: 218


training the regression head and finding the best regularisation strength alpha.

In [15]:
# Alpha candidates. these are log-spaced from very weak to very strong regularisation
ALPHAS = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 500.0]
 
# TimeSeriesSplit ensures each fold's validation data is strictly after its training data — respecting temporal order within cross-validation and eliminating temporal leakage
tscv = TimeSeriesSplit(n_splits=5)
 
ridge_model = RidgeCV(
    alphas=ALPHAS,
    cv=tscv,
    scoring="neg_mean_squared_error",
    fit_intercept=True,
)
 
ridge_model.fit(X_train_reg, y_train_reg)
 
print(f"  Best alpha selected: {ridge_model.alpha_}")
print(f"  Number of coefficients: {len(ridge_model.coef_)}")
 
# Quick sanity check on validation set
val_pred_reg = ridge_model.predict(X_val_reg)
val_rmse = np.sqrt(mean_squared_error(y_val_reg, val_pred_reg))
print(f"  Validation RMSE (sanity check): {val_rmse:.4f} vehicles")

  Best alpha selected: 0.001
  Number of coefficients: 23
  Validation RMSE (sanity check): 3.3561 vehicles


training the classification head

In [16]:
# C is the inverse of regularisation strength: small C = strong regularisation
C_VALUES    = [0.001, 0.01, 0.1, 1.0, 10.0]
L1_RATIOS   = [0.1, 0.5, 0.9]   # 0 = pure Ridge penalty, 1 = pure Lasso
 
lasso_model = LogisticRegressionCV(
    Cs=C_VALUES,
    cv=tscv,
    penalty="elasticnet",
    solver="saga",
    l1_ratios=L1_RATIOS,
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
 
lasso_model.fit(X_train_clf, y_train_clf)
 
print(f"  Best C selected   : {lasso_model.C_[0]:.4f}")
print(f"  Best l1_ratio     : {lasso_model.l1_ratio_[0]:.2f}  "
      f"(0=Ridge-only, 1=Lasso-only)")
 
# Quick validation Macro-F1
val_pred_clf = lasso_model.predict(X_val_clf)
val_f1 = f1_score(y_val_clf, val_pred_clf, average="macro", zero_division=0)
print(f"  Validation Macro-F1 (sanity check): {val_f1:.4f}")

  Best C selected   : 0.1000
  Best l1_ratio     : 0.90  (0=Ridge-only, 1=Lasso-only)
  Validation Macro-F1 (sanity check): 0.1978


generating test set predictions and evaluating all the four models, following the standard rollin-origin multi-horizon evaluation protocol

In [17]:

# Base predictions on the full test feature matrix
test_pred_reg_base  = ridge_model.predict(X_test)
test_pred_clf_base  = lasso_model.predict(X_test_clf)
test_pred_proba_base = lasso_model.predict_proba(X_test_clf)
 
# Helper: compute all metrics for one horizon 
def evaluate_horizon(pred_reg, true_reg, pred_clf, true_clf, pred_proba,
                     horizon_name, horizon_steps, results_dict):
    
    #Shifting predictions forward by horizon_steps to simulate forecasting h steps ahead, then compute RMSE, MAE, Macro-F1, PR-AUC.
    
    h = horizon_steps

    #Regression
    # pred_reg[i] predicts the value at time i+h.
    # Shift: drop last h predictions, drop first h ground-truth values.
    pred_r = pred_reg[:-h] if h > 0 else pred_reg
    true_r = true_reg[h:]  if h > 0 else true_reg
 
    # Drop any NaN in the regression target (10 missing vehicle_count rows)
    valid_mask_r = ~np.isnan(true_r)
    pred_r = pred_r[valid_mask_r]
    true_r = true_r[valid_mask_r]
 
    rmse = np.sqrt(mean_squared_error(true_r, pred_r))
    mae  = mean_absolute_error(true_r, pred_r)
 
    # Classification
    pred_c      = pred_clf[:-h]   if h > 0 else pred_clf
    true_c      = true_clf[h:]    if h > 0 else true_clf
    pred_proba_c = pred_proba[:-h] if h > 0 else pred_proba
 
    macro_f1 = f1_score(true_c, pred_c, average="macro", zero_division=0)
 
    # PR-AUC: average_precision_score with one-vs-rest for multiclass
    # We need at least 2 classes present in the slice for this to work
    present_classes = np.unique(true_c)
    if len(present_classes) >= 2:
        # One-vs-rest macro PR-AUC across all 5 congestion classes
        pr_auc = average_precision_score(
            true_c, pred_proba_c,
            average="macro",
        )
    else:
        pr_auc = float("nan")
 
    results_dict[horizon_name] = {
        "RMSE": rmse, "MAE": mae,
        "Macro-F1": macro_f1, "PR-AUC": pr_auc,
    }
 
    print(f"\n  [{horizon_name}]")
    print(f"    RMSE     : {rmse:.4f}")
    print(f"    MAE      : {mae:.4f}")
    print(f"    Macro-F1 : {macro_f1:.4f}")
    print(f"    PR-AUC   : {pr_auc:.4f}")
 
 
results_full = {}   # stores full model results for Section 12 comparison
 
for h_name, h_steps in HORIZONS.items():
    evaluate_horizon(
        pred_reg   = test_pred_reg_base,
        true_reg   = y_test_reg_full,
        pred_clf   = test_pred_clf_base,
        true_clf   = y_test_clf,
        pred_proba = test_pred_proba_base,
        horizon_name  = h_name,
        horizon_steps = h_steps,
        results_dict  = results_full,
    )


  in]
    RMSE     : 4.8840
    MAE      : 3.9078
    Macro-F1 : 0.1894
    PR-AUC   : 0.2248

  in]
    RMSE     : 4.9390
    MAE      : 3.9292
    Macro-F1 : 0.2115
    PR-AUC   : 0.2236

  in]
    RMSE     : 5.4294
    MAE      : 4.3843
    Macro-F1 : 0.1817
    PR-AUC   : 0.2190


Ablation - ridge without exogenous covariates

In [18]:
# Features to KEEP (endogenous only)
ENDOGENOUS_COLS = [
    "lag1", "lag2", "lag3", "lag6", "lag24", "roll3",
    "avg_speed",
    "hour_sin", "hour_cos",
    "weekday_sin", "weekday_cos",
]
ENDOGENOUS_COLS = [c for c in ENDOGENOUS_COLS if c in train_df.columns]
 
print(f"  Ablation feature set: {len(ENDOGENOUS_COLS)} columns (endogenous only)")
print(f"  Dropped: {set(FEATURE_COLS) - set(ENDOGENOUS_COLS)}")
 
# Re-scale with only the endogenous columns
scaler_abl = StandardScaler()
X_train_abl = scaler_abl.fit_transform(train_df[ENDOGENOUS_COLS])
X_val_abl   = scaler_abl.transform(val_df[ENDOGENOUS_COLS])
X_test_abl  = scaler_abl.transform(test_df[ENDOGENOUS_COLS])
 
X_train_clf_abl = X_train_abl[clf_train_mask.values]
X_test_clf_abl  = X_test_abl[clf_test_mask.values]
 
# Retrain regression head
ridge_abl = RidgeCV(alphas=ALPHAS, cv=tscv,
                    scoring="neg_mean_squared_error", fit_intercept=True)
ridge_abl.fit(X_train_abl[reg_train_mask.values], y_train_reg)
print(f"  Ablation ridge best alpha: {ridge_abl.alpha_}")
 
# Retrain classification head
lasso_abl = LogisticRegressionCV(
    Cs=C_VALUES, cv=tscv, penalty="elasticnet", solver="saga",
    l1_ratios=L1_RATIOS,
    class_weight="balanced", max_iter=2000,
    random_state=RANDOM_STATE, n_jobs=-1,
)
lasso_abl.fit(X_train_clf_abl, y_train_clf)
print(f"  Ablation lasso best C: {lasso_abl.C_[0]:.4f}")
 
# Predictions
test_pred_reg_abl   = ridge_abl.predict(X_test_abl)
test_pred_clf_abl_p = lasso_abl.predict(X_test_clf_abl)
test_pred_proba_abl = lasso_abl.predict_proba(X_test_clf_abl)
 
results_ablation = {}
 
print("\n  Ablation results (endogenous features only):")
for h_name, h_steps in HORIZONS.items():
    evaluate_horizon(
        pred_reg   = test_pred_reg_abl,
        true_reg   = y_test_reg_full,
        pred_clf   = test_pred_clf_abl_p,
        true_clf   = y_test_clf,
        pred_proba = test_pred_proba_abl,
        horizon_name  = h_name,
        horizon_steps = h_steps,
        results_dict  = results_ablation,
    )

  Ablation feature set: 11 columns (endogenous only)
  Dropped: {'road_Good', 'school_in_session', 'is_holiday', 'weather_Rainy', 'weather_Foggy', 'weather_Unknown', 'sensor_ok', 'event_flag', 'road_Poor', 'road_Moderate', 'weather_Clear', 'road_Unknown'}
  Ablation ridge best alpha: 0.001
  Ablation lasso best C: 0.1000

  Ablation results (endogenous features only):

  in]
    RMSE     : 4.8730
    MAE      : 3.9060
    Macro-F1 : 0.2152
    PR-AUC   : 0.2176

  in]
    RMSE     : 4.9418
    MAE      : 3.9342
    Macro-F1 : 0.2091
    PR-AUC   : 0.2329

  in]
    RMSE     : 5.4305
    MAE      : 4.3882
    Macro-F1 : 0.1724
    PR-AUC   : 0.2052


interpretability outputs and comparative results table.

In [19]:
# Ridge coefficient magnitude chart (regression head)
coef_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "coefficient": ridge_model.coef_,
    "abs_coef": np.abs(ridge_model.coef_)
}).sort_values("abs_coef", ascending=False).head(15)
 
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#1F5C99" if c >= 0 else "#C55A11" for c in coef_df["coefficient"]]
ax.barh(coef_df["feature"][::-1], coef_df["coefficient"][::-1], color=colors[::-1])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Ridge coefficient (standardised scale)", fontsize=11)
ax.set_title("Top 15 Ridge regression coefficients\n"
             "(blue = positive contribution, orange = negative)",
             fontsize=12, fontweight="bold")
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.savefig("ridge_coefficients.png", dpi=150)
plt.close()
print("  Saved: ridge_coefficients.png")
 
# Lasso sparsity report (classification head). LogisticRegressionCV with elasticnet gives one coef_ row per class.
# A feature is "sparse" if all its class-weights are effectively zero.
clf_coef_matrix = lasso_model.coef_   # shape: (n_classes, n_features)
feature_max_abs  = np.max(np.abs(clf_coef_matrix), axis=0)  # max across classes
 
sparsity_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "max_abs_coef_across_classes": feature_max_abs,
    "zeroed_out": feature_max_abs < 1e-6
}).sort_values("max_abs_coef_across_classes")
 
print("\n  Lasso sparsity report (features zeroed out by penalty):")
zeroed = sparsity_df[sparsity_df["zeroed_out"]]
if len(zeroed) > 0:
    for _, row in zeroed.iterrows():
        print(f"    ZEROED → {row['feature']}")
else:
    print("    No features fully zeroed (l1_ratio may favour Ridge blend)")
 
print("\n  Top 10 most influential classification features:")
top10 = sparsity_df.sort_values("max_abs_coef_across_classes", ascending=False).head(10)
for _, row in top10.iterrows():
    print(f"    {row['feature']:35s}  max|coef| = {row['max_abs_coef_across_classes']:.4f}")
 
#Confusion matrix 
# Use the 15-min horizon predictions (h=1 step) for the confusion matrix
h = HORIZONS["15min"]
pred_cm  = test_pred_clf_base[:-h] if h > 0 else test_pred_clf_base
true_cm  = y_test_clf[h:]          if h > 0 else y_test_clf
 
cm = confusion_matrix(true_cm, pred_cm, labels=[0, 1, 2, 3, 4])
cm_df = pd.DataFrame(cm,
                     index=[f"True {i}" for i in range(5)],
                     columns=[f"Pred {i}" for i in range(5)])
 
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues",
            linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Confusion matrix — congestion classification\n"
             "(Ridge/Lasso, 15-min horizon, test set)",
             fontsize=12, fontweight="bold")
ax.set_ylabel("True class", fontsize=11)
ax.set_xlabel("Predicted class", fontsize=11)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.close()
print("\n  Saved: confusion_matrix.png")
 
# Full vs. Ablation comparison table 
print("\n  ── Comparative results: Full Ridge vs. Ablation (no exogenous) ──")
print(f"\n  {'Metric':<12}", end="")
for h in HORIZONS:
    print(f"  {'Full ':>8} {'Abl ':>8}  {'Δ':>6}   ", end="")
    _ = h   # suppress unused warning
print()
 
header_printed = False
for metric in ["RMSE", "MAE", "Macro-F1", "PR-AUC"]:
    print(f"  {metric:<12}", end="")
    for h_name in HORIZONS:
        full_val = results_full[h_name][metric]
        abl_val  = results_ablation[h_name][metric]
        if not np.isnan(full_val) and not np.isnan(abl_val):
            delta = full_val - abl_val
            # For error metrics (RMSE, MAE): negative delta = full is better
            # For score metrics (F1, PR-AUC): positive delta = full is better
            arrow = ""
            if metric in ["RMSE", "MAE"]:
                arrow = "↓" if delta < 0 else "↑"
            else:
                arrow = "↑" if delta > 0 else "↓"
            print(f"  {full_val:>8.4f} {abl_val:>8.4f}  {delta:>+6.4f}{arrow}  ", end="")
        else:
            print(f"  {'N/A':>8} {'N/A':>8}  {'N/A':>7}   ", end="")
    print()
 
print("\n  Column headers for above table (per group of 3):")
for h_name in HORIZONS:
    print(f"    [{h_name}]  Full | Ablation | Delta")
 
#  Clean summary for each horizon
print("\n  ── Final test-set results summary (Full Ridge / Lasso model) ──")
summary_rows = []
for h_name, metrics in results_full.items():
    row = {"Horizon": h_name}
    row.update(metrics)
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows).set_index("Horizon")
print(summary_df.to_string(float_format=lambda x: f"{x:.4f}"))
 
summary_df.to_csv("ridge_results_full.csv")
print("\n  Saved: ridge_results_full.csv")
 
abl_rows = []
for h_name, metrics in results_ablation.items():
    row = {"Horizon": h_name}
    row.update(metrics)
    abl_rows.append(row)
abl_df = pd.DataFrame(abl_rows).set_index("Horizon")
abl_df.to_csv("ridge_results_ablation.csv")
print("  Saved: ridge_results_ablation.csv")
 
# Full classification report 
print("\n  ── Classification report (15-min horizon, test set) ──")
print(classification_report(true_cm, pred_cm,
                             target_names=[f"Class {i}" for i in range(5)],
                             zero_division=0))
 
#end 

print("Pipeline complete.  Output files:")
print("  ridge_coefficients.png   — top 15 Ridge regression coefficients")
print("  confusion_matrix.png     — classification confusion matrix (15-min)")
print("  ridge_results_full.csv   — full model metrics at all 3 horizons")
print("  ridge_results_ablation.csv — ablation model metrics at all 3 horizons")

  Saved: ridge_coefficients.png

  Lasso sparsity report (features zeroed out by penalty):
    ZEROED → lag3
    ZEROED → is_holiday
    ZEROED → school_in_session
    ZEROED → weather_Unknown

  Top 10 most influential classification features:
    lag2                                 max|coef| = 0.2129
    event_flag                           max|coef| = 0.1260
    weather_Foggy                        max|coef| = 0.0953
    sensor_ok                            max|coef| = 0.0888
    road_Unknown                         max|coef| = 0.0874
    lag24                                max|coef| = 0.0816
    hour_sin                             max|coef| = 0.0746
    road_Poor                            max|coef| = 0.0704
    hour_cos                             max|coef| = 0.0572
    road_Moderate                        max|coef| = 0.0565

  Saved: confusion_matrix.png

  ── Comparative results: Full Ridge vs. Ablation (no exogenous) ──

  Metric           Full      Abl        Δ        Full 

In [20]:
import joblib, os, json

os.makedirs("hf_deploy", exist_ok=True)

joblib.dump(ridge_model, "hf_deploy/ridge_model.pkl")
joblib.dump(lasso_model, "hf_deploy/lasso_model.pkl")
joblib.dump(scaler,      "hf_deploy/scaler.pkl")

with open("hf_deploy/feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f)

print("Done!")


Done!
